# 01 - Preparacao de DadosNotebook compartilhado para exploracao inicial, limpeza, analise de correlacoes e geracao dos datasets intermediarios.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from pathlib import Path

print("Imports OK")

## 2. Carregamento do Dataset

In [ ]:
import os

# â”€â”€ Detecta ambiente â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

    subprocess.run(["pip", "install", "kaggle", "-q"], check=True, capture_output=True)
    os.makedirs("/content/data", exist_ok=True)
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", "dhivyeshrk/diseases-and-symptoms-dataset",
        "--unzip", "-p", "/content/data/"
    ], check=True)
    DATA_PATH = Path("/content/data/Final_Augmented_dataset_Diseases_and_Symptoms.csv")
    OUTPUT_DIR = Path("/content/data")
else:
    # Resolve a raiz do projeto independente de onde o Jupyter foi iniciado
    project_root = Path.cwd()
    for _ in range(4):
        if (project_root / "data").is_dir():
            break
        project_root = project_root.parent
    else:
        raise FileNotFoundError(
            "DiretÃ³rio 'data/' nÃ£o encontrado. "
            "Inicie o Jupyter a partir da raiz do projeto ou de notebooks/individuais/."
        )

    DATA_PATH = project_root / "data" / "Final_Augmented_dataset_Diseases_and_Symptoms.csv"
    OUTPUT_DIR = project_root / "data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo nÃ£o encontrado: {DATA_PATH}\n"
        "Baixe o dataset do Kaggle e coloque-o na pasta 'data/' na raiz do projeto."
    )

# Carregamento em chunks para economizar memÃ³ria
# Apenas as colunas necessÃ¡rias sÃ£o mantidas em memÃ³ria
chunks = []
for chunk in pd.read_csv(DATA_PATH, chunksize=50_000, low_memory=False):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"Shape: {df.shape}")
print(f"Colunas: {len(df.columns)}")
print(f"MemÃ³ria usada: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Dados carregados de: {DATA_PATH}")

## 3. Primeira inspeÃ§Ã£o

In [ ]:
df.head()

In [ ]:
df.info()

## 4. ReduÃ§Ã£o de tipos de dados (otimizaÃ§Ã£o de memÃ³ria)

In [ ]:
mem_antes = df.memory_usage(deep=True).sum() / 1024**2
print(f"MemÃ³ria antes: {mem_antes:.1f} MB")

# Coluna target: category (economiza memÃ³ria em colunas de alta cardinalidade repetida)
df["diseases"] = df["diseases"].astype("category")

# Colunas de sintomas: 0/1 -> int8 (8x menor que int64)
sintoma_cols = df.columns[1:].tolist()
df[sintoma_cols] = df[sintoma_cols].apply(pd.to_numeric, downcast="integer")

mem_depois = df.memory_usage(deep=True).sum() / 1024**2
print(f"MemÃ³ria depois: {mem_depois:.1f} MB")
print(f"ReduÃ§Ã£o: {((mem_antes - mem_depois) / mem_antes * 100):.1f}%")
gc.collect()

## 5. VerificaÃ§Ã£o de valores nulos

In [ ]:
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

if len(nulos) == 0:
    print("Nenhum valor nulo encontrado.")
else:
    print(f"{len(nulos)} colunas com valores nulos:")
    print(nulos.head(20))

## 6. VerificaÃ§Ã£o de duplicatas

In [ ]:
duplicatas = df.duplicated().sum()
print(f"Linhas duplicadas: {duplicatas}")

if duplicatas > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Shape apÃ³s remoÃ§Ã£o: {df.shape}")

## 7. AnÃ¡lise bÃ¡sica dos dados

In [ ]:
print("=" * 60)
print("DOENÃ‡AS NO DATASET")
print("=" * 60)
print(f"Total de doenÃ§as Ãºnicas: {df['diseases'].nunique()}")
print(f"\nTop 15 doenÃ§as mais frequentes:")
print(df["diseases"].value_counts().head(15))

In [ ]:
print("=" * 60)
print("SINTOMAS")
print("=" * 60)
num_sintomas = df.shape[1] - 1  # exceto coluna 'diseases'
print(f"Total de colunas de sintomas: {num_sintomas}")

# ProporÃ§Ã£o de 1s em cada sintoma
prop = df.iloc[:, 1:].mean().sort_values(ascending=False)
print(f"\nTop 15 sintomas mais comuns:")
print(prop.head(15).round(3))

## 8. RemoÃ§Ã£o de sintomas raros e constantes

In [ ]:
# Identificar colunas com quase nenhuma variÃ¢ncia
prop = df.iloc[:, 1:].mean()

# Sintomas presentes em menos de 0.1% ou mais de 99.9% dos casos
sintomas_raros = prop[prop < 0.001].index.tolist()
sintomas_quase_constantes = prop[prop > 0.999].index.tolist()

print(f"Sintomas muito raros (< 0.1%): {len(sintomas_raros)}")
print(f"Sintomas quase constantes (> 99.9%): {len(sintomas_quase_constantes)}")

cols_remover = sintomas_raros + sintomas_quase_constantes
print(f"\nTotal de colunas a remover: {len(cols_remover)}")

In [ ]:
if cols_remover:
    df_limpo = df.drop(columns=cols_remover)
    print(f"Shape antes: {df.shape}")
    print(f"Shape depois: {df_limpo.shape}")
else:
    df_limpo = df.copy()
    print("Nenhuma coluna removida.")

## 9. AnÃ¡lise de correlaÃ§Ã£o entre sintomas

In [ ]:
# Amostra para correlaÃ§Ã£o (limita uso de memÃ³ria da matriz n_sintomas Ã— n_sintomas)
N_AMOSTRA = min(20_000, len(df_limpo))
amostra = df_limpo.sample(n=N_AMOSTRA, random_state=42)

# CorrelaÃ§Ã£o apenas entre sintomas (exclui 'diseases')
corr = amostra.iloc[:, 1:].astype("float32").corr(method="pearson")
del amostra
gc.collect()

print(f"Matriz de correlaÃ§Ã£o: {corr.shape}")
print(f"MemÃ³ria da matriz: {corr.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# Top correlaÃ§Ãµes positivas (excluindo diagonal)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
corr_vals = corr.where(mask).stack().sort_values(ascending=False)

print("Top 20 pares de sintomas mais correlacionados:")
for (s1, s2), val in corr_vals.head(20).items():
    print(f"  {val:.3f} -- {s1} <-> {s2}")

In [ ]:
# Heatmap das top correlaÃ§Ãµes
top_n = 30
top_cols = corr.abs().sum().sort_values(ascending=False).head(top_n).index
corr_top = corr.loc[top_cols, top_cols]

plt.figure(figsize=(14, 12))
sns.heatmap(corr_top, cmap="RdBu_r", center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title(f"CorrelaÃ§Ã£o entre os {top_n} sintomas mais relevantes", fontsize=14)
plt.tight_layout()
plt.show()

## 10. CorrelaÃ§Ã£o sintoma Ã— doenÃ§a

In [ ]:
# CorrelaÃ§Ã£o entre cada sintoma e as doenÃ§as mais frequentes
top_doencas = df_limpo["diseases"].value_counts().head(10).index
df_top = df_limpo[df_limpo["diseases"].isin(top_doencas)].copy()

if hasattr(df_top["diseases"], "cat"):
    df_top["diseases"] = df_top["diseases"].cat.remove_unused_categories()

dummies = pd.get_dummies(df_top["diseases"], prefix="doenca")
sintoma_cols_top = df_top.columns[1:].tolist()

# CorrelaÃ§Ã£o vetorizada via NumPy (muito mais rÃ¡pido que corrwith em loop)
X = df_top[sintoma_cols_top].values.astype(np.float32)   # (n_linhas, n_sintomas)
Y = dummies.values.astype(np.float32)                     # (n_linhas, n_doencas)

X_c = X - X.mean(axis=0)
Y_c = Y - Y.mean(axis=0)

X_std = X_c.std(axis=0)
Y_std = Y_c.std(axis=0)
X_std[X_std == 0] = np.nan  # evita divisÃ£o por zero em sintomas constantes

corr_matrix = (X_c.T @ Y_c) / (len(X) * X_std[:, None] * Y_std[None, :])
corr_disease = pd.DataFrame(corr_matrix, index=sintoma_cols_top, columns=dummies.columns)

del df_top, dummies, X, Y, X_c, Y_c, X_std, Y_std, corr_matrix
gc.collect()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_disease.astype(float), cmap="YlOrRd", linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("CorrelaÃ§Ã£o Sintoma x DoenÃ§a (top 10 doenÃ§as)", fontsize=14)
plt.tight_layout()
plt.show()

del corr_disease
gc.collect()

## 11. Resumo final e salvamento

In [ ]:
print("=" * 60)
print("RESUMO DO TRATAMENTO DE DADOS")
print("=" * 60)
print(f"Dataset original:      {df.shape[0]} linhas, {df.shape[1]} colunas")
print(f"ApÃ³s limpeza:          {df_limpo.shape[0]} linhas, {df_limpo.shape[1]} colunas")
print(f"Colunas removidas:     {df.shape[1] - df_limpo.shape[1]}")
print(f"Duplicatas removidas:  {df.duplicated().sum()}")
print(f"DoenÃ§as Ãºnicas:        {df_limpo['diseases'].nunique()}")
print(f"MemÃ³ria final:         {df_limpo.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# Salva o dataset tratado
OUTPUT_PATH = OUTPUT_DIR / "dataset_tratado.csv"
df_limpo.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset tratado salvo em: {OUTPUT_PATH}")

# Libera memÃ³ria ao fim
del df, df_limpo
gc.collect()
print("MemÃ³ria liberada.")

## 12. Escolha das 3 doencas e geracao do dataset filtrado

As doencas escolhidas sao **pneumonia**, **gout** e **anxiety**.
Criterios: volumes semelhantes (~1200 amostras cada), dominios clinicos distintos
(respiratorio, articular e psiquiatrico), o que torna o problema de classificacao
desafiador e pedagogicamente rico.

In [ ]:
import sys

# Adiciona raiz do projeto ao path para importar src.utils
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import CLASSES_ALVO, filter_diseases, save_dataset, FILTERED_DATA_PATH

print(f"Classes alvo: {CLASSES_ALVO}")

# Recarrega df_limpo se nao estiver em memoria (execucao parcial do notebook)
if "df_limpo" not in globals():
    import pandas as pd
    df_limpo = pd.read_csv(OUTPUT_DIR / "dataset_tratado.csv")
    print("df_limpo recarregado do CSV.")
else:
    print("df_limpo ja esta em memoria.")


In [ ]:
# Verifica volumes antes de filtrar
counts = df_limpo['diseases'].astype(str).value_counts()
print('Volumes das classes alvo:')
for cls in CLASSES_ALVO:
    print(f'  {cls}: {counts.get(cls, 0)} amostras')

# Filtra para as 3 doencas e remove colunas 100% zeradas pos-filtro
df_filtrado = filter_diseases(df_limpo, CLASSES_ALVO)

print()
print(f'Shape original (limpo): {df_limpo.shape}')
print(f'Shape apos filtro:      {df_filtrado.shape}')
print()
print('Distribuicao das classes:')
print(df_filtrado['diseases'].value_counts())

In [ ]:
# Salva o dataset filtrado
saved_path = save_dataset(df_filtrado, FILTERED_DATA_PATH)
print(f"Dataset filtrado salvo em: {saved_path}")
print(f"Arquivo gerado: {saved_path.stat().st_size / 1024:.1f} KB")

del df_filtrado
gc.collect()
print("Memoria liberada.")
